In [ ]:
import networkx as nx
from qarp import config
from qarp.graphs import Graph, Hypergraph, SimplicialComplex
from qarp.graphs import community_vector_to_sets, lists_to_tuples
from qarp.graphs import generate_complete_simplicial_complex, generate_random_simplicial_complex

config.seed = 1234

### Graphs

In [ ]:
edge_list = [[0, 1], [1, 2], [2, 3]]
G = Graph(edge_list)
print("Nodes: ", G.nodes)
print("Edges: ", G.edges)

print("----------------")

G.add_node(4)
print("Nodes: ", G.nodes)
G.add_edge(0, 4)
print("Edges: ", G.edges)

In [ ]:
# Sparse matrices are methods on the graph (append .toarray() for a dense one).
print("Adjacency matrix:\n", G.adjacency_matrix().toarray())
print("Degree matrix:\n", G.degree_matrix().toarray())
print("Laplacian matrix:\n", G.laplacian_matrix().toarray())

In [ ]:
print("Incidence matrix:\n", G.incidence_matrix().toarray())
print("Normalized Laplacian:\n", G.normalized_laplacian_matrix().toarray())
print("Modularity matrix (dense, from networkx):\n", nx.modularity_matrix(G))
print("G is connected: ", nx.is_connected(G))
print("Laplacian eigenvalues: ", nx.laplacian_spectrum(G))
G.plot()

In [ ]:
G2 = Graph([(1, 2), (2, 3), (3, 4)])
print("Modularity matrix:\n", nx.modularity_matrix(G2))
print("Max modularity eigenvector: ", G2.max_modularity_eigenvector())
print("Modularity of community vector [1, 1, -1, -1]: ", nx.community.modularity(G2, community_vector_to_sets(G2, [1, 1, -1, -1])))
print("Communities via greedy modularity: ", nx.community.greedy_modularity_communities(G2, best_n=2))

#### Cost Hamiltonian and QAOA

Node labels are qubit indices, so `n_qubits` is the highest label + 1 (not the node count).
`to_cost_hamiltonian` gives the Ising form $\sum_e w_e Z_i Z_j$ that QAOA minimises — not the
MaxCut objective; minimising it maximises the cut, with $\text{cut} = (\sum_e w_e - \langle H \rangle)/2$.

In [ ]:
from qarp.algorithms import QAOA

weighted = Graph()
weighted.add_edge(0, 1, weight=2.0)
weighted.add_edge(2, 3, weight=2.0)

H = weighted.to_cost_hamiltonian()
print("H =", H)
print("qubits:", weighted.n_qubits)

qaoa = QAOA(weighted, n_layers=2, verbose=False).build()
energy, params = qaoa.run()
total_weight = sum(w for _, _, w in weighted.edges(data="weight"))
print("min energy:", round(energy, 3), "-> cut:", round((total_weight - energy) / 2, 3))

`Graph.from_qubit_operator` is the inverse: $Z_i Z_j$ terms become weighted edges and $Z_i$ terms
the node attribute `linear`, which `to_cost_hamiltonian` reads back. The round trip closes up to
the constant term, which has no home on a graph.

In [ ]:
from qarp.operators import QubitOperator

op = QubitOperator("Z0 Z1", 1.5) + QubitOperator("Z1", 0.3) + QubitOperator("", 2.0)
back = Graph.from_qubit_operator(op)
print("edges:", list(back.edges(data="weight")))
print("linear:", dict(back.nodes(data="linear")))
print("round trip up to the constant:", back.to_cost_hamiltonian() == op - QubitOperator("", 2.0))

#### Graph state

`to_graph_state_block` prepares the graph state: a Hadamard on every qubit, then a CZ per edge.
For a single edge that is $(|00\rangle + |01\rangle + |10\rangle - |11\rangle)/2$.

In [ ]:
import numpy as np
import qarpx as qx

state = Graph([(0, 1)]).to_graph_state_block().build()
print(np.round(qx.QarpSimulator().statevector(state.flatten(), state.n_qubits), 3))
state.plot()

### Hypergraphs

In [ ]:
G3 = Hypergraph([(1, 2), (2, 3), (1, 2, 3)])
print([node for node in G3.nodes])
print([G3.edges[idx] for idx in G3.edges])
print("qubits:", G3.n_qubits)  # highest vertex + 1

# The hypergraph state: H on every qubit, then a C^(k-1)Z per hyperedge of order k.
G3.to_state_block().build().plot()
G3.plot()

### Simplicial complexes

In [ ]:
G4 = SimplicialComplex([(0, 1), (1, 2), (1, 2, 3), (3, 4)])

print("simplices:", G4.get_simplices())
print("vertices:", G4.vertices(), " neighbours of 1:", G4.neighbors(1))
print("cofaces of (1, 2):", G4.cofaces((1, 2)))
print("f-vector:", G4.f_vector(), " Euler characteristic:", G4.euler_characteristic())
print("1-skeleton edges:", list(G4.to_graph().edges))
print("boundary d1:\n", G4.boundary_matrix(1).toarray())
print("Hodge Laplacian L0:\n", G4.hodge_laplacian(0).toarray())
G4.plot()

### Utils

In [ ]:
lists_to_tuples([[1, 2], [3, 4], [5, 6, 7]])

In [ ]:
G5 = generate_complete_simplicial_complex(4)
print(G5.get_simplices())

In [ ]:
G6 = generate_random_simplicial_complex(4, 0.2)
print(G6.get_simplices())